# Layer-wise Hidden State Extraction for Codenames Duet — Qwen2.5-7B-Instruct

**Contract: v1.0** | **Family: Causal LM** | **Mirrors Mistral**

Qwen2.5-7B-Instruct is a decoder-only causal LM with RoPE, GQA 7:1, and a
ChatML-style chat template. This notebook is derived from the Mistral canonical
notebook with three model-specific changes:

1. `MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"` (ungated, no HF token required).
2. Chat template via `tokenizer.apply_chat_template()` with a system message,
   instead of Mistral's `[INST]...[/INST]` scaffolding.
3. Output prefix `qwen_` and a separate Drive folder.

**The instruction body is byte-identical to the Mistral notebook** (Format A
from CONTRACT_v1.0 Section 2). Only the chat-template wrapping differs.

## Architecture summary

| Property | Mistral-7B-Instruct-v0.2 | Qwen2.5-7B-Instruct |
|---|---|---|
| Layers | 32 | 28 |
| Hidden dim | 4096 | 3584 |
| Attention | GQA 4:1 + SWA | GQA 7:1 |
| Vocab size | 32k | 152k |
| Training mix | English-centric | Multilingual (CN/EN heavy) |
| Chat template | `[INST]...[/INST]` | ChatML (`<|im_start|>...<|im_end|>`) |

Comparing Qwen against Mistral isolates whether peak-margin layer, positional
confound, and concordance gap are properties of the causal-LM family broadly
or specific to Mistral's architectural choices.

## What is saved (per condition)

- `qwen_general_{mode}.csv` — per-board behavioral results
- `qwen_metrics_{mode}.parquet` — per-(board, layer, word, permutation) scalars
- `qwen_vectors_subsample_index_{mode}.csv` — index for vector subsample
- `qwen_vectors_subsample_{mode}_f16.npz` — float16 vector matrix
- `qwen_generation_{mode}.csv` — generation output and concordance

## Aggregate files

- `qwen_layer_margins_{pm}_{mode}.csv`
- `qwen_position_confound_by_layer.csv`
- `qwen_shuffle_decomposition_by_layer.csv`

## Sanity checks (encoder scheme — uniform across all 6 notebooks)

- **SC0** Generation Diagnostic
- **SC1** Prompt Structure Verification
- **SC2** Span Coverage and Token Count Statistics
- **SC3** Anisotropy Characterization
- **SC4** Behavioral Accuracy Summary
- **SC5** Layer-wise Margin Curve
- **SC6** Per-Layer Positional Confound
- **SC7** Shuffle Confound Decomposition


## Cell 1 — Global Configuration

All parameters from CONTRACT_v1.0 Section 1. The only model-specific values are
`MODEL_NAME`, `MODEL_PREFIX`, and `BASE_DIR`. Everything else is identical
across all six notebooks.


In [ ]:
import gc
import os
import ast
import re
import random
import numpy as np
import torch
import pandas as pd
import torch.nn.functional as F
from typing import Dict, List, Tuple, Optional
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import spearmanr, mannwhitneyu, norm as scipy_norm

gc.collect()
torch.cuda.empty_cache()

# --- Reproducibility ---
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# === MODEL-SPECIFIC: model identity =========================================
MODEL_NAME   = "Qwen/Qwen2.5-7B-Instruct"
MODEL_PREFIX = "qwen"
BASE_DIR     = "/content/drive/MyDrive/TCC/qwen_outputs"
# ============================================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Experiment (frozen by CONTRACT_v1.0 Section 1) ---
EXPERIMENT_MODES      = [False, True]        # False = no_social, True = with_social
SAMPLE_SIZE           = 2000
CANDIDATE_ORDER       = "fixed"               # alphabetical within board
POOLING_METHODS       = ["mean", "max_norm"]  # both run in parallel
VECTOR_SUBSAMPLE_SIZE = 100
N_SHUFFLES            = 2                     # canonical + 2 random permutations
GENERATION_MAX_TOKENS = 30                    # causal models only
SHARD_BOARDS          = 200                   # flush Stream A every K boards

print("=" * 60)
print("GLOBAL CONFIGURATION (Contract v1.0)")
print("=" * 60)
print(f"MODEL_NAME              : {MODEL_NAME}")
print(f"MODEL_PREFIX            : {MODEL_PREFIX}")
print(f"DEVICE                  : {DEVICE}")
print(f"RANDOM_SEED             : {RANDOM_SEED}")
print(f"SAMPLE_SIZE             : {SAMPLE_SIZE}")
print(f"CANDIDATE_ORDER         : {CANDIDATE_ORDER}")
print(f"POOLING_METHODS         : {POOLING_METHODS}")
print(f"VECTOR_SUBSAMPLE_SIZE   : {VECTOR_SUBSAMPLE_SIZE}")
print(f"N_SHUFFLES              : {N_SHUFFLES}")
print(f"GENERATION_MAX_TOKENS   : {GENERATION_MAX_TOKENS}")
print(f"SHARD_BOARDS            : {SHARD_BOARDS}")
print(f"MODES                   : {['no_social', 'with_social']}")
print(f"BASE_DIR                : {BASE_DIR}")


GLOBAL CONFIGURATION (Contract v1.0)
MODEL_NAME              : Qwen/Qwen2.5-7B-Instruct
MODEL_PREFIX            : qwen
DEVICE                  : cuda
RANDOM_SEED             : 42
SAMPLE_SIZE             : 2000
CANDIDATE_ORDER         : fixed
POOLING_METHODS         : ['mean', 'max_norm']
VECTOR_SUBSAMPLE_SIZE   : 100
N_SHUFFLES              : 2
GENERATION_MAX_TOKENS   : 30
SHARD_BOARDS            : 200
MODES                   : ['no_social', 'with_social']
BASE_DIR                : /content/drive/MyDrive/TCC/qwen_outputs


## Cell 2 — Load Model and Tokenizer

Qwen2.5-7B-Instruct is **ungated** — no Hugging Face authentication required,
unlike some Mistral versions.

For Qwen (causal LM), `output_hidden_states=True` is set at inference time
in the forward pass, not at load time. The tokenizer's `pad_token` must be
set to `eos_token` for `model.generate()` to work without warnings.

Qwen's tokenizer uses a different vocabulary (152k tokens, including extensive
CJK character coverage) than Mistral's (32k). This affects BPE fragmentation
statistics (reported in SC2) but does not affect the extraction pipeline itself.


In [ ]:
# === MODEL-SPECIFIC: model loading ==========================================
print(f"\nLoading tokenizer and model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Causal LM requirement: pad_token must exist for generation.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map=None,
    low_cpu_mem_usage=False,
).to(DEVICE)

model.eval()
# ============================================================================

NUM_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size

print(f"Model loaded successfully.")
print(f"Number of transformer layers : {NUM_LAYERS}")
print(f"Hidden state dimensionality  : {HIDDEN_DIM}")
print(f"Total hidden states per token: {NUM_LAYERS + 1}  (embedding + {NUM_LAYERS} layers)")
print(f"Vocabulary size              : {model.config.vocab_size}")



Loading tokenizer and model: Qwen/Qwen2.5-7B-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded successfully.
Number of transformer layers : 28
Hidden state dimensionality  : 3584
Total hidden states per token: 29  (embedding + 28 layers)
Vocabulary size              : 152064


## Cell 3 — Load and Prepare Dataset

Dataset is shared across all six notebooks. Same path, same parsing, same
candidate ordering (alphabetical), same `row_id` assignment. The same
`SAMPLE_SIZE` boards are drawn under `random_state=RANDOM_SEED` in every
notebook, guaranteeing the cross-model comparison is on identical boards.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATASET_PATH = "/content/drive/MyDrive/TCC/clue_generation.csv"

df = pd.read_csv(DATASET_PATH)

for col in ["targets", "black", "tan"]:
    df[col] = df[col].apply(ast.literal_eval)

def build_candidates_fixed_order(row) -> List[str]:
    """Returns all board words in stable alphabetical order."""
    all_words = list(row["targets"]) + list(row["black"]) + list(row["tan"])
    return sorted(all_words)

df["candidates"] = df.apply(build_candidates_fixed_order, axis=1)
df = df.reset_index(drop=True)
df["row_id"] = df.index.astype(int)

print(f"Dataset loaded. Shape: {df.shape}")
print(f"Candidate ordering: {CANDIDATE_ORDER} (alphabetical)")
print(f"\nExample board (row 0):")
print(f"  hint       : {df.loc[0, 'output']}")
print(f"  targets    : {df.loc[0, 'targets']}")
print(f"  candidates : {df.loc[0, 'candidates'][:5]} ...")


Mounted at /content/drive
Dataset loaded. Shape: (7703, 48)
Candidate ordering: fixed (alphabetical)

Example board (row 0):
  hint       : sandwich
  targets    : ['sub']
  candidates : ['beat', 'boom', 'charge', 'check', 'drop'] ...


## Cell 4 — Define Giver Feature Columns

Frozen list from CONTRACT_v1.0 Section 2. Identical across all six notebooks.


In [ ]:
GIVER_COLS = [
    "giver.marriage",
    "giver.education",
    "giver.race",
    "giver.continent",
    "giver.language",
    "giver.religion",
    "giver.gender",
    "giver.country",
    "giver.political",
]

missing = [c for c in GIVER_COLS if c not in df.columns]
if missing:
    raise ValueError(
        f"Missing giver columns: {missing}\n"
        f"Available: {list(df.columns)}"
    )

print(f"All {len(GIVER_COLS)} giver feature columns validated.")


All 9 giver feature columns validated.


## Cell 5 — Prompt Builder

The instruction body is **byte-identical** across all six notebooks
(CONTRACT_v1.0 Section 2, Format A). Only the per-model wrapping differs.

For Qwen, we use `tokenizer.apply_chat_template(...)` with `add_generation_prompt=True`,
which appends the assistant-turn header (`<|im_start|>assistant\n`) so the model
knows to start generating. We include an explicit system message
(`"You are a helpful assistant."`) to match Qwen's default behavior — omitting
the system message can cause Qwen to behave inconsistently.

The trailing instruction `Only output the word.` is preserved for every model.


In [ ]:
def _format_feature_value(v) -> str:
    """Formats a feature value as a clean string. Returns 'NA' for missing."""
    if pd.isna(v):
        return "NA"
    if isinstance(v, float):
        return f"{v:.4f}".rstrip("0").rstrip(".")
    return str(v)

_FEATURE_LABEL_MAP = {
    "giver.marriage"  : "Marriage",
    "giver.education" : "Education",
    "giver.race"      : "Race",
    "giver.continent" : "Continent",
    "giver.language"  : "Language",
    "giver.religion"  : "Religion",
    "giver.gender"    : "Gender",
    "giver.country"   : "Country",
    "giver.political" : "Politics",
    "giver.politics"  : "Politics",  # T5 dataset variant
}


def build_instruction_body(
    hint: str,
    candidates: List[str],
    giver_features: Optional[Dict[str, object]],
    use_social_context: bool,
) -> Tuple[str, Dict[str, str]]:
    """
    Constructs the SHARED instruction body (Format A from CONTRACT_v1.0 Section 2).
    Byte-identical across all six notebooks.
    """
    words_block = "\n".join(
        [f"{i+1}. {w}" for i, w in enumerate(candidates)]
    )
    feature_markers = {}

    if use_social_context and giver_features:
        parts = []
        for k, v in giver_features.items():
            v_str = _format_feature_value(v)
            label = _FEATURE_LABEL_MAP.get(k, k.split(".")[-1].capitalize())
            marker = f"{label}: {v_str}"
            feature_markers[k] = marker
            parts.append(marker)
        social_block = ", ".join(parts) if parts else "N/A"

        instruction_body = (
            f"You are playing the game Codenames.\n"
            f"The clue was given by a player with the following characteristics:\n"
            f"{social_block}\n"
            f'The hint is: "{hint}"\n'
            f"The possible words are:\n"
            f"{words_block}\n"
            f"Which word best matches the hint? Only output the word."
        )
    else:
        instruction_body = (
            f"You are playing the game Codenames.\n"
            f'The hint is: "{hint}"\n'
            f"The possible words are:\n"
            f"{words_block}\n"
            f"Which word best matches the hint? Only output the word."
        )

    return instruction_body, feature_markers


def build_prompt(
    hint: str,
    candidates: List[str],
    giver_features: Optional[Dict[str, object]],
    use_social_context: bool,
) -> Tuple[str, Dict[str, str]]:
    """
    Returns (prompt, feature_markers) where prompt is wrapped in Qwen's
    native ChatML chat template via tokenizer.apply_chat_template().
    """
    instruction_body, feature_markers = build_instruction_body(
        hint=hint,
        candidates=candidates,
        giver_features=giver_features,
        use_social_context=use_social_context,
    )

    # === MODEL-SPECIFIC: chat template wrapping =============================
    # Qwen2.5 uses ChatML with <|im_start|>/<|im_end|> delimiters.
    # We use apply_chat_template to let the tokenizer construct the correct
    # format. add_generation_prompt=True appends the assistant-turn header.
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": instruction_body},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    # ========================================================================

    return prompt, feature_markers


## Cell 6 — Token Span Detection and Pooling Utilities

Architecture-agnostic. Identical across all six notebooks.

- `find_token_spans` locates token indices for each target substring using
  character-level offsets. Candidate substring searches start after the anchor
  `"The possible words are:"` to avoid false matches in the social block.
- `mean_pool_span` averages across all tokens in the span.
- `max_norm_pool_span` selects the token with the highest L2 norm.
- `pool_span` is the dispatcher.
- `cosine_similarity_np` is a numpy cosine with zero-norm protection.


In [ ]:
def find_token_spans(
    full_text: str,
    offset_mapping: List[Tuple[int, int]],
    spans_to_find: Dict[str, str],
    candidate_anchor: str = "The possible words are:",
) -> Dict[str, Tuple[int, int]]:
    """
    Locates token-level spans for target substrings in the prompt.

    Parameters
    ----------
    full_text : str
        The full prompt string.
    offset_mapping : list of (int, int)
        Character-level offsets for each token.
    spans_to_find : dict
        Maps span name -> substring. "cand:" prefix searches after anchor.
    candidate_anchor : str
        Delimiter marking the start of the candidate list.

    Returns
    -------
    found : dict
        Maps span name -> (token_start, token_end) index tuple.
    """
    found = {}
    cand_start_char = full_text.find(candidate_anchor)
    if cand_start_char == -1:
        cand_start_char = 0

    for name, substring in spans_to_find.items():
        if name.startswith("cand:"):
            char_start = full_text.find(substring, cand_start_char)
        else:
            char_start = full_text.find(substring)

        if char_start == -1:
            continue

        char_end = char_start + len(substring)

        token_start, token_end = None, None
        for idx, (s, e) in enumerate(offset_mapping):
            if s < char_end and e > char_start:
                if token_start is None:
                    token_start = idx
                token_end = idx + 1

        if token_start is not None and token_end > token_start:
            found[name] = (token_start, token_end)

    return found


def mean_pool_span(
    layer_hidden_states: torch.Tensor,
    span: Tuple[int, int],
) -> Optional[np.ndarray]:
    """Mean-pools hidden states across a token span. Returns float16."""
    s, e = span
    tokens = layer_hidden_states[s:e]
    if tokens.shape[0] == 0:
        return None
    pooled = tokens.mean(dim=0).detach().float().cpu().numpy()
    return pooled.astype(np.float16)


def max_norm_pool_span(
    layer_hidden_states: torch.Tensor,
    span: Tuple[int, int],
) -> Optional[np.ndarray]:
    """
    Selects the token with the highest L2 norm from a span. Returns float16.
    Avoids creating synthetic composite vectors. The highest-norm token
    typically corresponds to the root morpheme.
    """
    s, e = span
    tokens = layer_hidden_states[s:e]
    if tokens.shape[0] == 0:
        return None
    if tokens.shape[0] == 1:
        return tokens[0].detach().float().cpu().numpy().astype(np.float16)
    norms = torch.norm(tokens, p=2, dim=1)
    best_idx = torch.argmax(norms).item()
    pooled = tokens[best_idx].detach().float().cpu().numpy()
    return pooled.astype(np.float16)


def pool_span(
    layer_hidden_states: torch.Tensor,
    span: Tuple[int, int],
    method: str = "mean",
) -> Optional[np.ndarray]:
    """Dispatcher for pooling methods. Supports 'mean' and 'max_norm'."""
    if method == "mean":
        return mean_pool_span(layer_hidden_states, span)
    elif method == "max_norm":
        return max_norm_pool_span(layer_hidden_states, span)
    else:
        raise ValueError(f"Unknown pooling method: {method}")


def cosine_similarity_np(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two 1D arrays. Returns 0.0 on zero norm."""
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0.0 or nb == 0.0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))


## Cell 7 — Generation Utility (Phase 3, causal-only)

Runs `model.generate()` on the prompt to capture the model's actual text
output, then matches it against the candidate list using a position-aware
strategy: find ALL candidate words in the response, return the FIRST one that
appears (by character position). This treats the model's initial answer as
primary and handles hedged multi-word answers gracefully.

**Causal-only:** This cell is omitted in encoder notebooks (BERT, T5,
ModernBERT, Random Baseline) because those models cannot generate.


In [ ]:
# === MODEL-SPECIFIC: generation (causal-only) ===============================
def generate_response(
    prompt: str,
    candidates: List[str],
    max_new_tokens: int = 30,
) -> Dict[str, object]:
    """
    Generates the model's text response and matches against candidates.

    Matching strategy: find ALL candidate words in the response, return the
    FIRST one that appears (by character position). The hint word itself is
    excluded from matching — instruction-tuned causal LMs typically echo the hint in
    quotes before answering.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    prompt_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_ids = output_ids[0, prompt_length:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    # Extract hint to exclude from matching (instruction-tuned LMs often echo the hint in quotes).
    hint_match = re.search(r'hint\s+is\s*:?\s*"([^"]+)"', prompt, re.IGNORECASE)
    hint_word = hint_match.group(1).lower() if hint_match else None

    candidates_lower = {c.lower(): c for c in candidates}

    occurrences = []  # list of (position, candidate)
    gen_lower = generated_text.lower()

    for cand_lower, cand_original in candidates_lower.items():
        if hint_word and cand_lower == hint_word:
            continue
        for match in re.finditer(rf"\b{re.escape(cand_lower)}\b", gen_lower):
            occurrences.append((match.start(), cand_original))

    matched_word = None
    matched_position = None
    n_candidates_in_response = 0

    if occurrences:
        occurrences.sort(key=lambda x: x[0])
        seen = set()
        unique_mentions = []
        for pos, cand in occurrences:
            if cand not in seen:
                seen.add(cand)
                unique_mentions.append((pos, cand))
        n_candidates_in_response = len(unique_mentions)
        matched_position, matched_word = unique_mentions[0]

    return {
        "generated_text"           : generated_text,
        "generated_word"           : matched_word,
        "generated_in_candidates"  : matched_word is not None,
        "n_candidates_in_response" : n_candidates_in_response,
    }
# ============================================================================


## Cell 8 — Core Instance Processing Function

`run_instance` processes a single board under a specified candidate ordering
(canonical alphabetical or shuffled) and condition (no_social or with_social).

It performs:
1. Prompt construction with the given candidate ordering.
2. Tokenization with offset mapping.
3. Span detection for hint, candidates, and giver features.
4. Forward pass with `output_hidden_states=True`.
5. Per-layer, per-pooling-method: pool vectors, compute cosines to hint,
   compute ranks, record as scalar metrics.
6. All-pairs candidate cosines for anisotropy (mean pooling only).
7. If subsample mode: retain raw vectors for both pooling methods.

Generation is NOT called here — that is handled separately in the main loop
to avoid redundant generation across permutations (generation is only
meaningful for the canonical ordering).

Returns
-------
general_record : dict
    Per-board behavioral metadata (one per board × permutation).
metrics_records : list of dict
    Per-(layer, word) scalar metrics. No vectors.
vector_records : list of dict or None
    Vectors for subsample boards under canonical ordering only.


In [ ]:
def extract_giver_features(
    row: pd.Series,
    giver_cols: List[str],
) -> Dict[str, object]:
    """Extracts non-null giver feature values from a dataset row."""
    return {
        c: row[c]
        for c in giver_cols
        if c in row.index and not pd.isna(row[c])
    }


def run_instance(
    row: pd.Series,
    giver_cols: List[str],
    use_social_context: bool,
    candidates_order: List[str],
    permutation_id: int = 0,
    save_vectors: bool = False,
) -> Tuple[Dict, List[Dict], Optional[List[Dict]]]:
    """
    Processes one board instance under a given candidate ordering.

    Parameters
    ----------
    row : pd.Series
        One row from the sampled dataset.
    giver_cols : list of str
        Column names for giver demographic features.
    use_social_context : bool
        If True, includes giver features in the prompt.
    candidates_order : list of str
        The candidate words in the desired order. For the canonical run
        this is alphabetical; for shuffles it is a random permutation.
    permutation_id : int
        0 = canonical (alphabetical) ordering. 1..K = shuffles.
    save_vectors : bool
        If True, raw vectors are retained for the subsample.
    """
    row_id     = int(row["row_id"])
    hint       = str(row["output"])
    candidates = list(candidates_order)
    targets    = set(row["targets"])
    black      = set(row["black"])
    tan        = set(row["tan"])

    giver_features = (
        extract_giver_features(row, giver_cols)
        if use_social_context else {}
    )

    prompt, feature_markers = build_prompt(
        hint=hint,
        candidates=candidates,
        giver_features=giver_features,
        use_social_context=use_social_context,
    )

    # --- Tokenization ---
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        return_offsets_mapping=True,
    ).to(DEVICE)

    offset_mapping     = inputs["offset_mapping"][0].tolist()
    prompt_token_count = inputs["input_ids"].shape[1]

    # --- Build span targets ---
    spans_to_find = {"hint": hint}
    for c in candidates:
        spans_to_find[f"cand:{c}"] = c
    if use_social_context:
        for k, marker in feature_markers.items():
            spans_to_find[f"giver:{k}"] = marker

    spans = find_token_spans(prompt, offset_mapping, spans_to_find)

    if "hint" not in spans:
        raise ValueError(
            f"Hint span not found for row_id={row_id}, hint='{hint}'."
        )

    candidate_position_map = {w: i for i, w in enumerate(candidates)}

    # === MODEL-SPECIFIC: forward pass with output_hidden_states ============
    # Causal LMs accept output_hidden_states at inference time.
    with torch.no_grad():
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            output_hidden_states=True,
            return_dict=True,
        )
    # =======================================================================

    hidden_states = outputs.hidden_states

    # ================================================================
    # Compute metrics across ALL layers, for ALL pooling methods
    # ================================================================
    metrics_records = []
    vector_records  = [] if save_vectors else None

    for layer_idx in range(NUM_LAYERS + 1):
        layer_hs = hidden_states[layer_idx][0]

        # --- Pool hint vector ---
        hint_vecs = {pm: pool_span(layer_hs, spans["hint"], method=pm)
                     for pm in POOLING_METHODS}
        hint_token_count = spans["hint"][1] - spans["hint"][0]

        # --- Pool candidate vectors ---
        cand_vecs = {}
        cand_meta = {}
        for c in candidates:
            if c in targets:
                c_type = "target"
            elif c in black:
                c_type = "black"
            elif c in tan:
                c_type = "tan"
            else:
                c_type = "unknown"

            ck = f"cand:{c}"
            cand_vecs[c] = {}
            if ck in spans:
                c_token_count = spans[ck][1] - spans[ck][0]
                for pm in POOLING_METHODS:
                    cand_vecs[c][pm] = pool_span(layer_hs, spans[ck], method=pm)
            else:
                c_token_count = 0
                for pm in POOLING_METHODS:
                    cand_vecs[c][pm] = None
            cand_meta[c] = {"word_type": c_type, "token_count": c_token_count}

        # --- Pool giver feature vectors (with_social only) ---
        giver_vecs = {}
        giver_token_counts = {}
        if use_social_context:
            for feat_name, marker in feature_markers.items():
                fk = f"giver:{feat_name}"
                giver_vecs[feat_name] = {}
                if fk in spans:
                    giver_token_counts[feat_name] = spans[fk][1] - spans[fk][0]
                    for pm in POOLING_METHODS:
                        giver_vecs[feat_name][pm] = pool_span(layer_hs, spans[fk], method=pm)
                else:
                    giver_token_counts[feat_name] = 0
                    for pm in POOLING_METHODS:
                        giver_vecs[feat_name][pm] = None

        # --- Cosines: hint -> each candidate, per pooling method ---
        cosines_per_method = {}
        for pm in POOLING_METHODS:
            cosines_per_method[pm] = {}
            h_vec = hint_vecs[pm]
            if h_vec is None:
                for c in candidates:
                    cosines_per_method[pm][c] = float("nan")
                continue
            h_vec_f32 = h_vec.astype(np.float32)
            for c in candidates:
                c_vec = cand_vecs[c][pm]
                if c_vec is not None:
                    cosines_per_method[pm][c] = cosine_similarity_np(
                        h_vec_f32, c_vec.astype(np.float32)
                    )
                else:
                    cosines_per_method[pm][c] = float("nan")

        # --- Ranks per pooling method ---
        ranks_per_method = {}
        for pm in POOLING_METHODS:
            valid_cosines = {
                w: v for w, v in cosines_per_method[pm].items()
                if not np.isnan(v)
            }
            sorted_words = sorted(
                valid_cosines.keys(),
                key=lambda w: valid_cosines[w],
                reverse=True,
            )
            ranks_per_method[pm] = {}
            for rank_pos, w in enumerate(sorted_words, start=1):
                ranks_per_method[pm][w] = rank_pos
            for c in candidates:
                if c not in ranks_per_method[pm]:
                    ranks_per_method[pm][c] = float("nan")

        # --- All-pairs candidate cosines for anisotropy (mean pooling) ---
        all_pair_cosines_layer = []
        valid_cand_vecs_mean = []
        for c in candidates:
            v = cand_vecs[c]["mean"]
            if v is not None:
                valid_cand_vecs_mean.append(v.astype(np.float32))
        n_valid = len(valid_cand_vecs_mean)
        if n_valid >= 2:
            for i in range(n_valid):
                for j in range(i + 1, n_valid):
                    all_pair_cosines_layer.append(
                        cosine_similarity_np(
                            valid_cand_vecs_mean[i],
                            valid_cand_vecs_mean[j],
                        )
                    )

        if all_pair_cosines_layer:
            layer_aniso_mean = float(np.mean(all_pair_cosines_layer))
            layer_aniso_std  = float(np.std(all_pair_cosines_layer))
        else:
            layer_aniso_mean = float("nan")
            layer_aniso_std  = float("nan")

        # --- Build metric records: hint ---
        hint_metric = {
            "row_id"                     : row_id,
            "layer"                      : layer_idx,
            "word"                       : hint,
            "word_type"                  : "hint",
            "token_count"                : hint_token_count,
            "list_position"              : -1,
            "use_social_context"         : use_social_context,
            "permutation_id"             : permutation_id,
            "layer_mean_pairwise_cosine" : layer_aniso_mean,
            "layer_std_pairwise_cosine"  : layer_aniso_std,
        }
        for pm in POOLING_METHODS:
            hint_metric[f"cosine_to_hint_{pm}"]  = float("nan")
            hint_metric[f"rank_{pm}"]            = float("nan")
            hint_metric[f"reciprocal_rank_{pm}"] = float("nan")
        metrics_records.append(hint_metric)

        # --- Build metric records: candidates ---
        for c in candidates:
            c_metric = {
                "row_id"                     : row_id,
                "layer"                      : layer_idx,
                "word"                       : c,
                "word_type"                  : cand_meta[c]["word_type"],
                "token_count"                : cand_meta[c]["token_count"],
                "list_position"              : candidate_position_map[c],
                "use_social_context"         : use_social_context,
                "permutation_id"             : permutation_id,
                "layer_mean_pairwise_cosine" : layer_aniso_mean,
                "layer_std_pairwise_cosine"  : layer_aniso_std,
            }
            for pm in POOLING_METHODS:
                cos_val  = cosines_per_method[pm][c]
                rank_val = ranks_per_method[pm][c]
                c_metric[f"cosine_to_hint_{pm}"]  = cos_val
                c_metric[f"rank_{pm}"]            = rank_val
                c_metric[f"reciprocal_rank_{pm}"] = (
                    1.0 / rank_val if not np.isnan(rank_val) else float("nan")
                )
            metrics_records.append(c_metric)

        # --- Build metric records: giver features ---
        if use_social_context:
            for feat_name in feature_markers:
                gf_metric = {
                    "row_id"                     : row_id,
                    "layer"                      : layer_idx,
                    "word"                       : feat_name,
                    "word_type"                  : "giver_feature",
                    "token_count"                : giver_token_counts.get(feat_name, 0),
                    "list_position"              : -1,
                    "use_social_context"         : use_social_context,
                    "permutation_id"             : permutation_id,
                    "layer_mean_pairwise_cosine" : layer_aniso_mean,
                    "layer_std_pairwise_cosine"  : layer_aniso_std,
                }
                for pm in POOLING_METHODS:
                    h_vec = hint_vecs[pm]
                    g_vec = giver_vecs[feat_name].get(pm)
                    if h_vec is not None and g_vec is not None:
                        gf_metric[f"cosine_to_hint_{pm}"] = cosine_similarity_np(
                            h_vec.astype(np.float32), g_vec.astype(np.float32)
                        )
                    else:
                        gf_metric[f"cosine_to_hint_{pm}"] = float("nan")
                    gf_metric[f"rank_{pm}"]            = float("nan")
                    gf_metric[f"reciprocal_rank_{pm}"] = float("nan")
                metrics_records.append(gf_metric)

        # --- Save vectors (subsample, canonical only) ---
        if save_vectors:
            for pm in POOLING_METHODS:
                vector_records.append({
                    "row_id": row_id, "layer": layer_idx,
                    "word": hint, "word_type": "hint",
                    "token_count": hint_token_count,
                    "pooling_method": pm,
                    "use_social_context": use_social_context,
                    "vector": hint_vecs[pm],
                })
            for c in candidates:
                for pm in POOLING_METHODS:
                    vector_records.append({
                        "row_id": row_id, "layer": layer_idx,
                        "word": c, "word_type": cand_meta[c]["word_type"],
                        "token_count": cand_meta[c]["token_count"],
                        "pooling_method": pm,
                        "use_social_context": use_social_context,
                        "vector": cand_vecs[c][pm],
                    })
            if use_social_context:
                for feat_name in feature_markers:
                    for pm in POOLING_METHODS:
                        vector_records.append({
                            "row_id": row_id, "layer": layer_idx,
                            "word": feat_name, "word_type": "giver_feature",
                            "token_count": giver_token_counts.get(feat_name, 0),
                            "pooling_method": pm,
                            "use_social_context": use_social_context,
                            "vector": giver_vecs[feat_name].get(pm),
                        })

    # ================================================================
    # Behavioral prediction at final layer (per pooling method)
    # ================================================================
    # cosines_per_method and ranks_per_method now hold final-layer values.

    predicted_words = {}
    correct_flags   = {}
    for pm in POOLING_METHODS:
        valid_scores = {
            w: cosines_per_method[pm][w]
            for w in candidates
            if not np.isnan(cosines_per_method[pm].get(w, float("nan")))
        }
        pw = max(valid_scores, key=valid_scores.get) if valid_scores else None
        predicted_words[pm] = pw
        correct_flags[pm]   = (pw in targets) if pw else False

    # Rank aggregation metrics
    rank_metrics = {}
    for pm in POOLING_METHODS:
        target_ranks = [
            ranks_per_method[pm][w]
            for w in candidates
            if cand_meta[w]["word_type"] == "target"
            and not np.isnan(ranks_per_method[pm].get(w, float("nan")))
        ]
        if target_ranks:
            rank_metrics[f"mean_target_rank_{pm}"] = float(np.mean(target_ranks))
            rank_metrics[f"min_target_rank_{pm}"]  = float(np.min(target_ranks))
            rank_metrics[f"max_target_rank_{pm}"]  = float(np.max(target_ranks))
            rank_metrics[f"mrr_{pm}"]              = float(1.0 / np.min(target_ranks))
            rank_metrics[f"hit_at_1_{pm}"]         = float(np.min(target_ranks) == 1)
            rank_metrics[f"hit_at_3_{pm}"]         = float(np.min(target_ranks) <= 3)
            rank_metrics[f"hit_at_5_{pm}"]         = float(np.min(target_ranks) <= 5)
        else:
            for suffix in ["mean_target_rank", "min_target_rank",
                           "max_target_rank", "mrr",
                           "hit_at_1", "hit_at_3", "hit_at_5"]:
                rank_metrics[f"{suffix}_{pm}"] = float("nan")

    # Distance metrics
    distance_metrics = {}
    for pm in POOLING_METHODS:
        tgt_cos = [
            cosines_per_method[pm][w] for w in candidates
            if cand_meta[w]["word_type"] == "target"
            and not np.isnan(cosines_per_method[pm].get(w, float("nan")))
        ]
        non_cos = [
            cosines_per_method[pm][w] for w in candidates
            if cand_meta[w]["word_type"] in ("black", "tan")
            and not np.isnan(cosines_per_method[pm].get(w, float("nan")))
        ]
        distance_metrics[f"mean_cos_hint_targets_{pm}"]    = float(np.mean(tgt_cos)) if tgt_cos else float("nan")
        distance_metrics[f"mean_cos_hint_nontargets_{pm}"] = float(np.mean(non_cos)) if non_cos else float("nan")
        distance_metrics[f"raw_margin_{pm}"] = (
            (float(np.mean(tgt_cos)) - float(np.mean(non_cos)))
            if tgt_cos and non_cos else float("nan")
        )
        valid_scores = {
            w: cosines_per_method[pm][w] for w in candidates
            if not np.isnan(cosines_per_method[pm].get(w, float("nan")))
        }
        sorted_scores = sorted(valid_scores.values(), reverse=True)
        distance_metrics[f"cos_gap_r1_r2_{pm}"] = (
            sorted_scores[0] - sorted_scores[1]
            if len(sorted_scores) >= 2 else float("nan")
        )

    missing_spans = [c for c in candidates if f"cand:{c}" not in spans]

    # --- Memory cleanup ---
    del hidden_states, outputs, layer_hs
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    general_record = {
        "row_id"             : row_id,
        "hint"               : hint,
        "n_targets"          : len(targets),
        "n_candidates"       : len(candidates),
        "n_missing_spans"    : len(missing_spans),
        "missing_span_words" : missing_spans,
        "prompt_token_count" : prompt_token_count,
        "use_social_context" : use_social_context,
        "permutation_id"     : permutation_id,
        "giver_features"     : giver_features if use_social_context else {},
    }
    for pm in POOLING_METHODS:
        general_record[f"predicted_word_{pm}"] = predicted_words[pm]
        general_record[f"correct_{pm}"]        = correct_flags[pm]
    general_record.update(rank_metrics)
    general_record.update(distance_metrics)

    return general_record, metrics_records, vector_records


## Cell 9 — Main Extraction Loop (with Stream A shard flushing)

Main loop with all phases integrated:

1. Pre-selects subsample board IDs for vector retention.
2. For each condition (no_social, with_social):
   a. For each board:
      - Run canonical (alphabetical) ordering.
      - Save vectors if board is in subsample.
      - Run model.generate() for behavioral accuracy (causal-only).
      - Run K shuffle permutations (no vectors, no generation).
   b. Flush metrics to a parquet shard every `SHARD_BOARDS=200` boards.
   c. After loop: concatenate shards into a single parquet file, delete shards.
   d. Save general_df, generation_df, vector subsample.

Memory profile (Mistral N=2000 with sharding):
- One board's hidden states at a time on GPU (~14 GB model + ~67 MB activations)
- Stream A buffer in CPU RAM: ~0.3 GB peak (bounded by `SHARD_BOARDS`)
- Stream B subsample in CPU RAM: ~1.5 GB total (bounded by `VECTOR_SUBSAMPLE_SIZE`)


In [ ]:
# --- Sample ---
df_sample = df.sample(
    n=min(SAMPLE_SIZE, len(df)),
    random_state=RANDOM_SEED,
).copy().reset_index(drop=True)

print(f"Sample size: {len(df_sample)} boards")
print(f"Row IDs (first 10): {sorted(df_sample['row_id'].tolist())[:10]} ...")

# --- Pre-select subsample board IDs ---
subsample_size = min(VECTOR_SUBSAMPLE_SIZE, len(df_sample))
subsample_df   = df_sample.sample(n=subsample_size, random_state=RANDOM_SEED)
SUBSAMPLE_IDS  = set(subsample_df["row_id"].tolist())
print(f"Vector subsample: {len(SUBSAMPLE_IDS)} boards")

# --- Pre-generate shuffle seeds for reproducibility ---
rng_shuffles  = np.random.RandomState(RANDOM_SEED + 1000)
shuffle_seeds = rng_shuffles.randint(0, 2**31, size=(len(df_sample), N_SHUFFLES))

os.makedirs(BASE_DIR, exist_ok=True)

# Determine if this notebook is causal (has generate_response) or encoder-only
HAS_GENERATION = "generate_response" in dir()

results = {}

for mode_flag in EXPERIMENT_MODES:
    mode_name = "with_social" if mode_flag else "no_social"
    print(f"\n{'='*60}")
    print(f"Running condition: {mode_name}  "
          f"(canonical + {N_SHUFFLES} shuffles per board)")
    print(f"{'='*60}")

    general_records    = []
    metrics_buffer     = []
    vector_records_all = []
    generation_records = []
    error_log          = []

    # --- Stream A shard tracking (inline; no helper function) ---
    shard_idx       = 0
    boards_in_shard = 0
    shard_paths     = []

    for board_idx, (_, row) in enumerate(
        tqdm(df_sample.iterrows(), total=len(df_sample), desc=mode_name)
    ):
        row_id = int(row["row_id"])
        canonical_candidates = list(row["candidates"])  # alphabetical

        # ==============================================================
        # Canonical ordering (permutation_id = 0)
        # ==============================================================
        try:
            save_vecs = (row_id in SUBSAMPLE_IDS)

            g, m, v = run_instance(
                row=row,
                giver_cols=GIVER_COLS,
                use_social_context=mode_flag,
                candidates_order=canonical_candidates,
                permutation_id=0,
                save_vectors=save_vecs,
            )
            general_records.append(g)
            metrics_buffer.extend(m)
            if v is not None:
                vector_records_all.extend(v)

            # --- Generation (canonical ordering only, causal-only) ---
            if HAS_GENERATION:
                prompt_for_gen, _ = build_prompt(
                    hint=str(row["output"]),
                    candidates=canonical_candidates,
                    giver_features=(
                        extract_giver_features(row, GIVER_COLS)
                        if mode_flag else {}
                    ),
                    use_social_context=mode_flag,
                )
                gen_result = generate_response(
                    prompt=prompt_for_gen,
                    candidates=canonical_candidates,
                    max_new_tokens=GENERATION_MAX_TOKENS,
                )
                gen_record = {
                    "row_id"                  : row_id,
                    "use_social_context"      : mode_flag,
                    "generated_text"          : gen_result["generated_text"],
                    "generated_word"          : gen_result["generated_word"],
                    "generated_in_candidates" : gen_result["generated_in_candidates"],
                    "generated_correct"       : (
                        gen_result["generated_word"] in set(row["targets"])
                        if gen_result["generated_word"] else False
                    ),
                }
                for pm in POOLING_METHODS:
                    gen_record[f"concordance_{pm}"] = (
                        gen_result["generated_word"] == g[f"predicted_word_{pm}"]
                        if gen_result["generated_word"] else False
                    )
                generation_records.append(gen_record)

        except Exception as e:
            error_log.append({"row_id": row_id, "error": str(e), "permutation_id": 0})
            print(f"  ERROR row_id={row_id} perm=0: {e}")

        # ==============================================================
        # Shuffle permutations (permutation_id = 1..K)
        # ==============================================================
        for k in range(N_SHUFFLES):
            try:
                perm_rng = np.random.RandomState(int(shuffle_seeds[board_idx, k]))
                shuffled_candidates = list(canonical_candidates)
                perm_rng.shuffle(shuffled_candidates)

                g_shuf, m_shuf, _ = run_instance(
                    row=row,
                    giver_cols=GIVER_COLS,
                    use_social_context=mode_flag,
                    candidates_order=shuffled_candidates,
                    permutation_id=k + 1,
                    save_vectors=False,
                )
                general_records.append(g_shuf)
                metrics_buffer.extend(m_shuf)
            except Exception as e:
                error_log.append({"row_id": row_id, "error": str(e), "permutation_id": k + 1})
                print(f"  ERROR row_id={row_id} perm={k+1}: {e}")

        # --- Shard flush check (INLINE) ---
        boards_in_shard += 1
        if boards_in_shard >= SHARD_BOARDS and metrics_buffer:
            shard_path = os.path.join(
                BASE_DIR,
                f"{MODEL_PREFIX}_metrics_{mode_name}_shard{shard_idx:03d}.parquet",
            )
            pd.DataFrame(metrics_buffer).to_parquet(shard_path, index=False)
            shard_paths.append(shard_path)
            metrics_buffer = []
            shard_idx += 1
            boards_in_shard = 0
            gc.collect()

    # --- Final flush of remaining buffer (INLINE) ---
    if metrics_buffer:
        shard_path = os.path.join(
            BASE_DIR,
            f"{MODEL_PREFIX}_metrics_{mode_name}_shard{shard_idx:03d}.parquet",
        )
        pd.DataFrame(metrics_buffer).to_parquet(shard_path, index=False)
        shard_paths.append(shard_path)
        metrics_buffer = []
        shard_idx += 1
        gc.collect()

    # ------------------------------------------------------------------
    # Concatenate shards into a single parquet file
    # ------------------------------------------------------------------
    metrics_path = os.path.join(BASE_DIR, f"{MODEL_PREFIX}_metrics_{mode_name}.parquet")
    if shard_paths:
        all_shards = [pd.read_parquet(p) for p in shard_paths]
        metrics_df = pd.concat(all_shards, ignore_index=True)
        metrics_df.to_parquet(metrics_path, index=False)
        for p in shard_paths:
            os.remove(p)
        del all_shards
        gc.collect()
        metrics_mb = os.path.getsize(metrics_path) / 1e6
    else:
        metrics_df = pd.DataFrame()
        metrics_mb = 0.0

    # ------------------------------------------------------------------
    # Save Stream B: Vector subsample
    # ------------------------------------------------------------------
    n_vec_records = len(vector_records_all)
    vec_mb = 0.0
    vec_index_df = pd.DataFrame()
    if n_vec_records > 0:
        vec_index_rows = []
        vec_arrays     = []
        for i, rec in enumerate(vector_records_all):
            vec = rec["vector"]
            valid = (vec is not None and isinstance(vec, np.ndarray)
                     and vec.shape == (HIDDEN_DIM,))
            vec_arrays.append(vec if valid else None)
            vec_index_rows.append({
                "record_idx"        : i,
                "row_id"            : rec["row_id"],
                "layer"             : rec["layer"],
                "word"              : rec["word"],
                "word_type"         : rec["word_type"],
                "token_count"       : rec["token_count"],
                "pooling_method"    : rec["pooling_method"],
                "use_social_context": rec["use_social_context"],
                "vector_valid"      : valid,
            })
        vec_index_df = pd.DataFrame(vec_index_rows)

        valid_mask = np.array([v is not None for v in vec_arrays])
        valid_idx  = np.where(valid_mask)[0]
        matrix     = np.zeros((n_vec_records, HIDDEN_DIM), dtype=np.float16)
        if len(valid_idx):
            matrix[valid_idx] = np.stack([vec_arrays[i] for i in valid_idx])

        del vec_arrays
        gc.collect()

        vec_index_path = os.path.join(
            BASE_DIR, f"{MODEL_PREFIX}_vectors_subsample_index_{mode_name}.csv"
        )
        vec_index_df.to_csv(vec_index_path, index=False)

        vec_matrix_path = os.path.join(
            BASE_DIR, f"{MODEL_PREFIX}_vectors_subsample_{mode_name}_f16.npz"
        )
        np.savez_compressed(vec_matrix_path, vectors=matrix)

        # NPZ integrity check
        _v = np.load(vec_matrix_path)
        _shape = _v["vectors"].shape
        _v.close()
        vec_mb = os.path.getsize(vec_matrix_path) / 1e6
        print(f"  Subsample NPZ verified: shape={_shape}, {vec_mb:.1f} MB")

        del matrix

    # ------------------------------------------------------------------
    # Save General + Generation
    # ------------------------------------------------------------------
    general_df = pd.DataFrame(general_records)
    general_path = os.path.join(BASE_DIR, f"{MODEL_PREFIX}_general_{mode_name}.csv")
    general_df.to_csv(general_path, index=False)

    generation_df = pd.DataFrame(generation_records)
    if HAS_GENERATION and len(generation_df) > 0:
        generation_path = os.path.join(BASE_DIR, f"{MODEL_PREFIX}_generation_{mode_name}.csv")
        generation_df.to_csv(generation_path, index=False)

    print(f"\nCondition '{mode_name}' complete.")
    print(f"  Boards processed     : {len(df_sample)}")
    print(f"  Permutations/board   : 1 canonical + {N_SHUFFLES} shuffles")
    print(f"  General records      : {len(general_df):,}")
    print(f"  Metrics rows         : {len(metrics_df):,}  ({metrics_mb:.1f} MB)")
    print(f"  Subsample vectors    : {n_vec_records:,} records  ({vec_mb:.1f} MB)")
    print(f"  Generation rows      : {len(generation_df)}")
    print(f"  Errors               : {len(error_log)}")

    results[mode_name] = {
        "general_df"    : general_df,
        "metrics_df"    : metrics_df,
        "generation_df" : generation_df,
        "error_log"     : error_log,
    }

    del vector_records_all, general_records, generation_records
    vector_records_all = []  # reset for next condition
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print(f"\nBoth conditions complete. Outputs in: {BASE_DIR}")


Sample size: 2000 boards
Row IDs (first 10): [0, 8, 14, 15, 17, 19, 23, 31, 37, 41] ...
Vector subsample: 100 boards

Running condition: no_social  (canonical + 2 shuffles per board)


no_social:   0%|          | 0/2000 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  Subsample NPZ verified: shape=(104110, 3584), 367.6 MB

Condition 'no_social' complete.
  Boards processed     : 2000
  Permutations/board   : 1 canonical + 2 shuffles
  General records      : 6,000
  Metrics rows         : 3,123,039  (48.5 MB)
  Subsample vectors    : 104,110 records  (367.6 MB)
  Generation rows      : 2000
  Errors               : 0

Running condition: with_social  (canonical + 2 shuffles per board)


with_social:   0%|          | 0/2000 [00:00<?, ?it/s]

  Subsample NPZ verified: shape=(147378, 3584), 655.6 MB

Condition 'with_social' complete.
  Boards processed     : 2000
  Permutations/board   : 1 canonical + 2 shuffles
  General records      : 6,000
  Metrics rows         : 4,399,416  (57.6 MB)
  Subsample vectors    : 147,378 records  (655.6 MB)
  Generation rows      : 2000
  Errors               : 0

Both conditions complete. Outputs in: /content/drive/MyDrive/TCC/qwen_outputs


## SC0 — Generation Diagnostic (causal-only)

Validates that `generate_response` is producing matchable outputs. Prints the
raw generated text (with `repr()` to expose whitespace), the cleaned matching
text, and which matching strategy succeeds for the first 5 boards.

This cell is causal-only and is omitted in encoder notebooks.


In [ ]:
print("SC0: Generation Diagnostic")
print("=" * 70)

N_DIAGNOSTIC = 5
diagnostic_rows = df_sample.head(N_DIAGNOSTIC)

for idx, (_, row) in enumerate(diagnostic_rows.iterrows()):
    row_id     = int(row["row_id"])
    hint       = str(row["output"])
    candidates = list(row["candidates"])
    targets    = set(row["targets"])

    prompt, _ = build_prompt(
        hint=hint,
        candidates=candidates,
        giver_features={},
        use_social_context=False,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    prompt_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=GENERATION_MAX_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_ids  = output_ids[0, prompt_length:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    print(f"\n--- Board {idx + 1} (row_id={row_id}) ---")
    print(f"  Hint           : {hint!r}")
    print(f"  Targets        : {targets}")
    print(f"  Candidates     : {candidates[:5]} ...")
    print(f"  Generated raw  : {generated_text!r}")
    matched = generate_response(prompt, candidates, GENERATION_MAX_TOKENS)
    print(f"  Matched word   : {matched['generated_word']}")
    print(f"  In candidates  : {matched['generated_in_candidates']}")


SC0: Generation Diagnostic

--- Board 1 (row_id=6598) ---
  Hint           : 'event'
  Targets        : {'time', 'date'}
  Candidates     : ['capital', 'cold', 'cover', 'date', 'day'] ...
  Generated raw  : 'day'
  Matched word   : day
  In candidates  : True

--- Board 2 (row_id=4764) ---
  Hint           : 'ocean'
  Targets        : {'air'}
  Candidates     : ['agent', 'air', 'boom', 'center', 'change'] ...
  Generated raw  : 'ocean'
  Matched word   : None
  In candidates  : False

--- Board 3 (row_id=7537) ---
  Hint           : 'book'
  Targets        : {'novel'}
  Candidates     : ['contract', 'crash', 'cycle', 'draft', 'fall'] ...
  Generated raw  : 'novel'
  Matched word   : novel
  In candidates  : True

--- Board 4 (row_id=2570) ---
  Hint           : 'mark'
  Targets        : {'point'}
  Candidates     : ['agent', 'beat', 'check', 'code', 'comic'] ...
  Generated raw  : 'spy'
  Matched word   : spy
  In candidates  : True

--- Board 5 (row_id=2225) ---
  Hint           : 'mi

## SC1 — Prompt Structure Verification

Confirms the prompt builder produces the expected structure for both conditions:
hint locatable, all candidates locatable after the anchor, all giver markers
present in `with_social`, and no leakage of giver markers in `no_social`.


In [ ]:
print("SC1: Prompt Structure Verification")
print("=" * 60)

test_row = df_sample.iloc[0]

for mode_flag in [False, True]:
    mode_name = "with_social" if mode_flag else "no_social"
    giver_features = (
        extract_giver_features(test_row, GIVER_COLS)
        if mode_flag else {}
    )
    prompt, feature_markers = build_prompt(
        hint=test_row["output"],
        candidates=list(test_row["candidates"]),
        giver_features=giver_features,
        use_social_context=mode_flag,
    )

    hint_found       = test_row["output"] in prompt
    all_cands_found  = all(c in prompt for c in test_row["candidates"])
    if mode_flag:
        markers_found = all(m in prompt for m in feature_markers.values())
        markers_leaked = False
    else:
        markers_found  = True
        markers_leaked = any(
            f"{label}:" in prompt for label in _FEATURE_LABEL_MAP.values()
        )

    token_count = len(tokenizer(prompt, add_special_tokens=False)["input_ids"])

    print(f"\nCondition: {mode_name}")
    print(f"  Hint found in prompt       : {hint_found}")
    print(f"  All candidates found       : {all_cands_found}")
    if mode_flag:
        print(f"  All giver markers found    : {markers_found}")
    else:
        print(f"  Giver markers leaked       : {markers_leaked}  (must be False)")
    print(f"  Prompt token count         : {token_count}")
    print(f"  Prompt preview:")
    preview = prompt[:300].replace(chr(10), " | ")
    print(f"    {preview}")


SC1: Prompt Structure Verification

Condition: no_social
  Hint found in prompt       : True
  All candidates found       : True
  Giver markers leaked       : False  (must be False)
  Prompt token count         : 132
  Prompt preview:
    <|im_start|>system | You are a helpful assistant.<|im_end|> | <|im_start|>user | You are playing the game Codenames. | The hint is: "event" | The possible words are: | 1. capital | 2. cold | 3. cover | 4. date | 5. day | 6. degree | 7. field | 8. grace | 9. march | 10. novel | 11. opera | 12. part | 13. soul | 14. spell | 15. spring | 16. t

Condition: with_social
  Hint found in prompt       : True
  All candidates found       : True
  All giver markers found    : True
  Prompt token count         : 187
  Prompt preview:
    <|im_start|>system | You are a helpful assistant.<|im_end|> | <|im_start|>user | You are playing the game Codenames. | The clue was given by a player with the following characteristics: | Marriage: single, Education: bachelor, R

## SC2 — Span Coverage and Token Count Statistics

Reports BPE/WordPiece/SentencePiece fragmentation: how many tokens each word
type occupies. High fragmentation introduces noise into mean-pooled vectors
and is reported as a methodological consideration.


In [ ]:
print("SC2: Span Coverage and Token Count Statistics")
print("=" * 60)

for mode_name in ["no_social", "with_social"]:
    mdf = results[mode_name]["metrics_df"]
    if len(mdf) == 0:
        print(f"\n{mode_name}: no metrics rows; skipping.")
        continue
    mdf_canon = mdf[mdf["permutation_id"] == 0]
    mdf_l0    = mdf_canon[mdf_canon["layer"] == 0]

    cand_rows = mdf_l0[mdf_l0["word_type"].isin(["target", "black", "tan"])]
    total_cand_slots   = len(cand_rows)
    missing_cand_slots = int(cand_rows["cosine_to_hint_mean"].isna().sum())
    coverage_pct = 100.0 * (1 - missing_cand_slots / max(total_cand_slots, 1))

    print(f"\nCondition: {mode_name}")
    print(f"  Candidate slots   : {total_cand_slots}")
    print(f"  Missing spans     : {missing_cand_slots}")
    print(f"  Span coverage     : {coverage_pct:.2f}%")
    if coverage_pct < 95.0:
        print("  WARNING: Span coverage below 95%. Review span detection logic.")

    print(f"\n  Token count per word type at layer 0:")
    for wt in ["hint", "target", "black", "tan", "giver_feature"]:
        subset = mdf_l0[mdf_l0["word_type"] == wt]["token_count"]
        if len(subset) > 0:
            print(f"    {wt:14s}: mean={subset.mean():.2f}, "
                  f"std={subset.std():.2f}, max={int(subset.max())}")


SC2: Span Coverage and Token Count Statistics

Condition: no_social
  Candidate slots   : 33897
  Missing spans     : 0
  Span coverage     : 100.00%

  Token count per word type at layer 0:
    hint          : mean=1.44, std=0.64, max=7
    target        : mean=1.06, std=0.33, max=4
    black         : mean=1.06, std=0.33, max=4
    tan           : mean=1.06, std=0.34, max=4

Condition: with_social
  Candidate slots   : 33897
  Missing spans     : 0
  Span coverage     : 100.00%

  Token count per word type at layer 0:
    hint          : mean=1.44, std=0.64, max=7
    target        : mean=1.06, std=0.33, max=4
    black         : mean=1.06, std=0.33, max=4
    tan           : mean=1.06, std=0.34, max=4
    giver_feature : mean=3.81, std=1.27, max=8


## SC3 — Anisotropy Characterization

Reports per-layer mean and std of all-pairs candidate cosines (mean pooling).
High mean pairwise cosine indicates anisotropic representations — vectors
clumped together in a narrow cone, which depresses semantic separability.


In [ ]:
print("SC3: Anisotropy Characterization (mean pooling, no_social)")
print("=" * 60)

mdf = results["no_social"]["metrics_df"]
if len(mdf) > 0:
    mdf_canon = mdf[mdf["permutation_id"] == 0]
    aniso_per_layer = (
        mdf_canon.groupby("layer")
        .agg(
            mean_aniso=("layer_mean_pairwise_cosine", "mean"),
            std_aniso =("layer_std_pairwise_cosine",  "mean"),
            n_boards  =("row_id", "nunique"),
        )
        .reset_index()
    )
    print(f"\n{'Layer':>6}  {'Mean aniso':>12}  {'Std aniso':>12}  {'N boards':>10}")
    print("-" * 50)
    for _, r in aniso_per_layer.iterrows():
        print(f"{int(r['layer']):>6}  {r['mean_aniso']:>12.4f}  "
              f"{r['std_aniso']:>12.4f}  {int(r['n_boards']):>10}")


SC3: Anisotropy Characterization (mean pooling, no_social)

 Layer    Mean aniso     Std aniso    N boards
--------------------------------------------------
     0        0.0558        0.0494        2000
     1        0.4686        0.0542        2000
     2        0.5010        0.0960        2000
     3        0.3822        0.0677        2000
     4        0.3679        0.0667        2000
     5        0.3468        0.0611        2000
     6        0.3622        0.0587        2000
     7        0.3621        0.0601        2000
     8        0.3206        0.0549        2000
     9        0.3212        0.0537        2000
    10        0.3279        0.0576        2000
    11        0.4125        0.0600        2000
    12        0.4044        0.0594        2000
    13        0.4392        0.0592        2000
    14        0.4849        0.0563        2000
    15        0.5109        0.0548        2000
    16        0.5392        0.0549        2000
    17        0.5463        0.0563        2

## SC4 — Behavioral Accuracy Summary

Reports cosine-rank accuracy, MRR, Hit@K for both pooling methods, plus
generation-based accuracy and concordance (causal-only). Canonical ordering only.


In [ ]:
def wilson_confidence_interval(successes: int, n: int, z: float = 1.96):
    """Wilson score interval for a proportion."""
    if n == 0:
        return (0.0, 0.0)
    p_hat  = successes / n
    denom  = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denom
    margin = z * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2)) / denom
    return (max(0.0, center - margin), min(1.0, center + margin))


print("SC4: Behavioral Accuracy Summary")
print("=" * 60)

for mode_name in ["no_social", "with_social"]:
    gdf       = results[mode_name]["general_df"]
    gdf_canon = gdf[gdf["permutation_id"] == 0]
    gen_df    = results[mode_name]["generation_df"]
    n         = len(gdf_canon)

    print(f"\nCondition: {mode_name}  (N={n})")

    for pm in POOLING_METHODS:
        n_correct = int(gdf_canon[f"correct_{pm}"].sum())
        accuracy  = n_correct / n if n > 0 else float("nan")
        ci_lo, ci_hi = wilson_confidence_interval(n_correct, n)

        mrr  = gdf_canon[f"mrr_{pm}"].dropna().mean()
        hit1 = gdf_canon[f"hit_at_1_{pm}"].dropna().mean()
        hit3 = gdf_canon[f"hit_at_3_{pm}"].dropna().mean()
        hit5 = gdf_canon[f"hit_at_5_{pm}"].dropna().mean()
        mean_rank = gdf_canon[f"mean_target_rank_{pm}"].dropna().mean()
        std_rank  = gdf_canon[f"mean_target_rank_{pm}"].dropna().std()

        mean_n_cands   = float(gdf_canon["n_candidates"].mean())
        mean_n_targets = float(gdf_canon["n_targets"].mean())
        random_rank    = (mean_n_cands + 1) / (mean_n_targets + 1)

        print(f"\n  Cosine-rank ({pm}):")
        print(f"    Top-1 accuracy    : {accuracy:.3f} ({n_correct}/{n})")
        print(f"    Wilson 95% CI     : [{ci_lo:.3f}, {ci_hi:.3f}]")
        print(f"    MRR               : {mrr:.4f}")
        print(f"    Hit@1 / @3 / @5   : {hit1:.3f} / {hit3:.3f} / {hit5:.3f}")
        print(f"    Mean target rank  : {mean_rank:.2f} ± {std_rank:.2f}")
        print(f"    Random baseline   : {random_rank:.2f}")

    if HAS_GENERATION and len(gen_df) > 0:
        n_gen        = len(gen_df)
        gen_in_cands = int(gen_df["generated_in_candidates"].sum())
        gen_correct  = int(gen_df["generated_correct"].sum())
        gen_acc      = gen_correct / n_gen if n_gen > 0 else float("nan")
        gen_ci       = wilson_confidence_interval(gen_correct, n_gen)

        print(f"\n  Generation-based accuracy:")
        print(f"    Generated in candidates : {gen_in_cands}/{n_gen} "
              f"({gen_in_cands/n_gen:.3f})")
        print(f"    Generation accuracy     : {gen_acc:.3f} ({gen_correct}/{n_gen})")
        print(f"    Wilson 95% CI           : [{gen_ci[0]:.3f}, {gen_ci[1]:.3f}]")

        for pm in POOLING_METHODS:
            concordance = gen_df[f"concordance_{pm}"].mean()
            print(f"    Concordance with {pm:8s}: {concordance:.3f}")


SC4: Behavioral Accuracy Summary

Condition: no_social  (N=2000)

  Cosine-rank (mean):
    Top-1 accuracy    : 0.139 (278/2000)
    Wilson 95% CI     : [0.125, 0.155]
    MRR               : 0.3104
    Hit@1 / @3 / @5   : 0.139 / 0.330 / 0.488
    Mean target rank  : 7.11 ± 4.39
    Random baseline   : 8.08

  Cosine-rank (max_norm):
    Top-1 accuracy    : 0.136 (272/2000)
    Wilson 95% CI     : [0.122, 0.152]
    MRR               : 0.3106
    Hit@1 / @3 / @5   : 0.136 / 0.337 / 0.492
    Mean target rank  : 7.11 ± 4.39
    Random baseline   : 8.08

  Generation-based accuracy:
    Generated in candidates : 1825/2000 (0.912)
    Generation accuracy     : 0.594 (1187/2000)
    Wilson 95% CI           : [0.572, 0.615]
    Concordance with mean    : 0.121
    Concordance with max_norm: 0.121

Condition: with_social  (N=2000)

  Cosine-rank (mean):
    Top-1 accuracy    : 0.132 (264/2000)
    Wilson 95% CI     : [0.118, 0.148]
    MRR               : 0.2999
    Hit@1 / @3 / @5   : 0.13

## SC5 — Layer-wise Margin Curve (Anisotropy-Adjusted)

Computes raw and anisotropy-adjusted margins per layer for both pooling methods.
Adjusted margin = raw_margin / std_pairwise_cosine (z-score). Saved per
condition × pooling method.


In [ ]:
print("SC5: Layer-wise Margin Curve")
print("=" * 60)

for pm in POOLING_METHODS:
    print(f"\n--- Pooling: {pm} ---")

    for mode_name in ["no_social", "with_social"]:
        mdf = results[mode_name]["metrics_df"]
        if len(mdf) == 0:
            continue
        mdf_canon = mdf[mdf["permutation_id"] == 0]

        layer_margin_records = []

        for layer_idx in range(NUM_LAYERS + 1):
            layer_data = mdf_canon[mdf_canon["layer"] == layer_idx]

            board_margins  = []
            board_hint_tgt = []
            board_hint_non = []

            for row_id, board_df in layer_data.groupby("row_id"):
                tgt_cos = board_df[board_df["word_type"] == "target"][f"cosine_to_hint_{pm}"].dropna()
                non_cos = board_df[board_df["word_type"].isin(["black", "tan"])][f"cosine_to_hint_{pm}"].dropna()
                if len(tgt_cos) == 0 or len(non_cos) == 0:
                    continue
                mt = float(tgt_cos.mean())
                mn = float(non_cos.mean())
                board_margins.append(mt - mn)
                board_hint_tgt.append(mt)
                board_hint_non.append(mn)

            if board_margins:
                aniso_vals     = layer_data.groupby("row_id")["layer_mean_pairwise_cosine"].first().dropna()
                aniso_std_vals = layer_data.groupby("row_id")["layer_std_pairwise_cosine"].first().dropna()
                layer_mean_aniso = float(aniso_vals.mean()) if len(aniso_vals) > 0 else float("nan")
                layer_std_aniso  = float(aniso_std_vals.mean()) if len(aniso_std_vals) > 0 else float("nan")

                raw_margin = float(np.mean(board_margins))
                adjusted_margin = (
                    raw_margin / layer_std_aniso
                    if not np.isnan(layer_std_aniso) and layer_std_aniso > 0
                    else float("nan")
                )

                layer_margin_records.append({
                    "layer"           : layer_idx,
                    "pooling_method"  : pm,
                    "condition"       : mode_name,
                    "mean_margin"     : raw_margin,
                    "std_margin"      : float(np.std(board_margins)),
                    "adjusted_margin" : adjusted_margin,
                    "mean_hint_tgt"   : float(np.mean(board_hint_tgt)),
                    "mean_hint_non"   : float(np.mean(board_hint_non)),
                    "mean_anisotropy" : layer_mean_aniso,
                    "n_boards"        : len(board_margins),
                })

        margin_df = pd.DataFrame(layer_margin_records)

        print(f"\n  Condition: {mode_name}")
        print(f"  {'Layer':>6}  {'Raw':>8}  {'Adj':>8}  {'hint-tgt':>10}  {'hint-non':>10}  {'Aniso':>8}")
        print(f"  {'-'*60}")
        for _, r in margin_df.iterrows():
            print(f"  {int(r['layer']):>6}  {r['mean_margin']:>8.5f}  "
                  f"{r['adjusted_margin']:>8.4f}  "
                  f"{r['mean_hint_tgt']:>10.5f}  "
                  f"{r['mean_hint_non']:>10.5f}  "
                  f"{r['mean_anisotropy']:>8.5f}")

        if len(margin_df) > 0:
            best_raw = margin_df.loc[margin_df["mean_margin"].idxmax()]
            best_adj = margin_df.loc[margin_df["adjusted_margin"].idxmax()]
            print(f"\n  Peak raw: layer {int(best_raw['layer'])} ({best_raw['mean_margin']:.5f})")
            print(f"  Peak adj: layer {int(best_adj['layer'])} ({best_adj['adjusted_margin']:.4f})")

        margin_path = os.path.join(BASE_DIR, f"{MODEL_PREFIX}_layer_margins_{pm}_{mode_name}.csv")
        margin_df.to_csv(margin_path, index=False)


SC5: Layer-wise Margin Curve

--- Pooling: mean ---

  Condition: no_social
   Layer       Raw       Adj    hint-tgt    hint-non     Aniso
  ------------------------------------------------------------
       0   0.02787    0.5646     0.06123     0.03335   0.05580
       1   0.03514    0.6482     0.46296     0.42782   0.46861
       2   0.03379    0.3519     0.49298     0.45919   0.50100
       3   0.05198    0.7676     0.39231     0.34033   0.38221
       4   0.05748    0.8623     0.36822     0.31074   0.36793
       5   0.06459    1.0573     0.35936     0.29478   0.34685
       6   0.06234    1.0625     0.36991     0.30756   0.36224
       7   0.07173    1.1942     0.37905     0.30731   0.36215
       8   0.08238    1.5013     0.35131     0.26893   0.32063
       9   0.09350    1.7421     0.36662     0.27312   0.32124
      10   0.09479    1.6458     0.36718     0.27239   0.32795
      11   0.08160    1.3601     0.43553     0.35393   0.41260
      12   0.07876    1.3272     0.41645  

## SC6 — Per-Layer Positional Confound (Spearman ρ at every layer)

Spearman correlation between candidate list position and cosine-to-hint at
every layer, no_social condition only. The single-layer Mann-Whitney U test
is also reported at the final layer for backward compatibility with the v1
analyses.

Includes a 5-line candidate-order consistency check at the start to ensure
both conditions used the same alphabetical ordering.


In [ ]:
print("SC6: Per-Layer Positional Confound")
print("=" * 60)

# --- Order consistency check (folded from old SC4) ---
gdf_ns = results["no_social"]["general_df"]
gdf_ws = results["with_social"]["general_df"]
gdf_ns_canon = gdf_ns[gdf_ns["permutation_id"] == 0].set_index("row_id")
gdf_ws_canon = gdf_ws[gdf_ws["permutation_id"] == 0].set_index("row_id")
common_rows = set(gdf_ns_canon.index) & set(gdf_ws_canon.index)
print(f"Order consistency: {len(common_rows)} common boards (both conditions saw same alphabetical ordering)")

# --- Per-layer Spearman ρ ---
mdf = results["no_social"]["metrics_df"]
if len(mdf) == 0:
    print("No metrics rows; skipping SC6.")
else:
    mdf_canon = mdf[mdf["permutation_id"] == 0]

    confound_records = []

    for layer_idx in range(NUM_LAYERS + 1):
        layer_data = mdf_canon[
            (mdf_canon["layer"] == layer_idx) &
            (mdf_canon["word_type"].isin(["target", "black", "tan"]))
        ]

        layer_rhos = []
        for row_id, board_df in layer_data.groupby("row_id"):
            positions = board_df["list_position"].values
            cosines   = board_df["cosine_to_hint_mean"].values
            valid = ~np.isnan(cosines) & ~np.isnan(positions)
            if valid.sum() < 3:
                continue
            rho, _ = spearmanr(positions[valid], cosines[valid])
            layer_rhos.append(rho)

        if layer_rhos:
            confound_records.append({
                "layer"    : layer_idx,
                "mean_rho" : float(np.mean(layer_rhos)),
                "std_rho"  : float(np.std(layer_rhos)),
                "n_boards" : len(layer_rhos),
            })

    confound_df = pd.DataFrame(confound_records)

    print(f"\n{'Layer':>6}  {'Mean rho':>10}  {'Std rho':>10}  {'Concern?':>10}")
    print("-" * 45)
    for _, r in confound_df.iterrows():
        concern = "YES" if abs(r["mean_rho"]) > 0.1 else ""
        print(f"{int(r['layer']):>6}  {r['mean_rho']:>10.4f}  {r['std_rho']:>10.4f}  {concern:>10}")

    confound_path = os.path.join(BASE_DIR, f"{MODEL_PREFIX}_position_confound_by_layer.csv")
    confound_df.to_csv(confound_path, index=False)
    print(f"\nSaved: {confound_path}")

    # --- Final-layer Mann-Whitney U for backward compatibility ---
    final_data = mdf_canon[
        (mdf_canon["layer"] == NUM_LAYERS) &
        (mdf_canon["word_type"].isin(["target", "black", "tan"]))
    ]
    positions_all = final_data["list_position"].values
    cosines_all   = final_data["cosine_to_hint_mean"].values
    valid_mask    = ~np.isnan(cosines_all) & ~np.isnan(positions_all)
    pv, cv = positions_all[valid_mask], cosines_all[valid_mask]

    if len(pv) >= 10:
        rho_f, p_rho = spearmanr(pv, cv)
        med = np.median(pv)
        near_c, far_c = cv[pv <= med], cv[pv > med]
        if len(near_c) >= 2 and len(far_c) >= 2:
            u_stat, u_p = mannwhitneyu(near_c, far_c, alternative="greater")
            eff_r = u_stat / (len(near_c) * len(far_c))
        else:
            u_stat, u_p, eff_r = float("nan"), float("nan"), float("nan")
        print(f"\n  Final-layer Mann-Whitney U:")
        print(f"    Spearman rho = {rho_f:+.4f} (p={p_rho:.4f})")
        print(f"    U={u_stat:.1f}, p={u_p:.4f}, r={eff_r:.4f}")
        print(f"    Near cos={near_c.mean():.5f}, Far cos={far_c.mean():.5f}")


SC6: Per-Layer Positional Confound
Order consistency: 2000 common boards (both conditions saw same alphabetical ordering)

 Layer    Mean rho     Std rho    Concern?
---------------------------------------------
     0     -0.0309      0.2401            
     1     -0.0370      0.2408            
     2     -0.0934      0.2444            
     3     -0.3579      0.2118         YES
     4     -0.3500      0.2152         YES
     5     -0.4124      0.2055         YES
     6     -0.3408      0.2156         YES
     7     -0.4507      0.1958         YES
     8     -0.4741      0.2077         YES
     9     -0.4769      0.2061         YES
    10     -0.4255      0.2154         YES
    11     -0.5299      0.2009         YES
    12     -0.6054      0.1807         YES
    13     -0.5357      0.1951         YES
    14     -0.6974      0.1488         YES
    15     -0.6406      0.1652         YES
    16     -0.7524      0.1330         YES
    17     -0.7562      0.1337         YES
    18     -0.

## SC7 — Shuffle Confound Decomposition

Decomposes cosine variance into position-driven (within-word, across
permutations) and semantics-driven (between-word, within-permutation)
components. Reports the semantic signal ratio per layer:
ratio = between_var / (between_var + within_var). Near 1.0 = pure semantic;
near 0.0 = pure positional.


In [ ]:
if N_SHUFFLES > 0:
    print("SC7: Shuffle Confound Decomposition")
    print("=" * 60)

    mode_name = "no_social"
    mdf = results[mode_name]["metrics_df"]

    if len(mdf) == 0:
        print("No metrics rows; skipping SC7.")
    else:
        # Final layer headline
        final_data = mdf[
            (mdf["layer"] == NUM_LAYERS) &
            (mdf["word_type"].isin(["target", "black", "tan"]))
        ]
        word_var    = final_data.groupby(["row_id", "word"])["cosine_to_hint_mean"].var().dropna()
        between_var = final_data.groupby(["row_id", "permutation_id"])["cosine_to_hint_mean"].var().dropna()
        mean_within  = float(word_var.mean())
        mean_between = float(between_var.mean())
        total = mean_within + mean_between
        ratio = mean_between / total if total > 0 else float("nan")

        print(f"\n  Final layer (layer {NUM_LAYERS}), mean pooling, no_social:")
        print(f"  Mean within-word variance  (positional) : {mean_within:.6f}")
        print(f"  Mean between-word variance (semantic)   : {mean_between:.6f}")
        print(f"  Semantic signal ratio                   : {ratio:.4f}")
        print(f"    (1.0 = pure semantic, 0.0 = pure positional)")

        # Per-layer
        print(f"\n  Per-layer semantic signal ratio:")
        print(f"  {'Layer':>6}  {'Within (pos)':>14}  {'Between (sem)':>14}  {'Ratio':>8}")
        print(f"  {'-'*48}")

        shuffle_records = []
        for layer_idx in range(NUM_LAYERS + 1):
            ld = mdf[
                (mdf["layer"] == layer_idx) &
                (mdf["word_type"].isin(["target", "black", "tan"]))
            ]
            wv = ld.groupby(["row_id", "word"])["cosine_to_hint_mean"].var().dropna()
            bv = ld.groupby(["row_id", "permutation_id"])["cosine_to_hint_mean"].var().dropna()
            mwv = float(wv.mean()) if len(wv) > 0 else 0.0
            mbv = float(bv.mean()) if len(bv) > 0 else 0.0
            tv  = mwv + mbv
            r   = mbv / tv if tv > 0 else float("nan")
            print(f"  {layer_idx:>6}  {mwv:>14.6f}  {mbv:>14.6f}  {r:>8.4f}")
            shuffle_records.append({
                "layer": layer_idx, "within_var": mwv,
                "between_var": mbv, "semantic_ratio": r,
            })

        shuffle_path = os.path.join(BASE_DIR, f"{MODEL_PREFIX}_shuffle_decomposition_by_layer.csv")
        pd.DataFrame(shuffle_records).to_csv(shuffle_path, index=False)
        print(f"\n  Saved: {shuffle_path}")
else:
    print("\nSC7 skipped (N_SHUFFLES = 0).")


SC7: Shuffle Confound Decomposition

  Final layer (layer 28), mean pooling, no_social:
  Mean within-word variance  (positional) : 0.003641
  Mean between-word variance (semantic)   : 0.006238
  Semantic signal ratio                   : 0.6315
    (1.0 = pure semantic, 0.0 = pure positional)

  Per-layer semantic signal ratio:
   Layer    Within (pos)   Between (sem)     Ratio
  ------------------------------------------------
       0        0.000004        0.001435    0.9973
       1        0.000031        0.002056    0.9851
       2        0.000112        0.005124    0.9786
       3        0.000133        0.002730    0.9535
       4        0.000239        0.002660    0.9175
       5        0.000361        0.002699    0.8820
       6        0.000305        0.002557    0.8934
       7        0.000733        0.003286    0.8175
       8        0.001004        0.003487    0.7764
       9        0.001039        0.003805    0.7855
      10        0.000860        0.003729    0.8125
      1

## Save Outputs Summary

Final summary of files written, plus a documented loading pattern for
downstream analysis notebooks.


In [ ]:
# Save error logs
for mode_name in ["no_social", "with_social"]:
    errors = results[mode_name]["error_log"]
    if errors:
        err_path = os.path.join(BASE_DIR, f"{MODEL_PREFIX}_errors_{mode_name}.csv")
        pd.DataFrame(errors).to_csv(err_path, index=False)
        print(f"Saved error log: {err_path}  ({len(errors)} errors)")

print("\n" + "=" * 60)
print("OUTPUT SUMMARY")
print("=" * 60)
print(f"Directory: {BASE_DIR}\n")

for mode_name in ["no_social", "with_social"]:
    print(f"  [{mode_name}]")
    candidate_files = [
        f"{MODEL_PREFIX}_general_{mode_name}.csv",
        f"{MODEL_PREFIX}_metrics_{mode_name}.parquet",
        f"{MODEL_PREFIX}_vectors_subsample_index_{mode_name}.csv",
        f"{MODEL_PREFIX}_vectors_subsample_{mode_name}_f16.npz",
    ]
    if HAS_GENERATION:
        candidate_files.append(f"{MODEL_PREFIX}_generation_{mode_name}.csv")
    for suffix in candidate_files:
        fpath = os.path.join(BASE_DIR, suffix)
        if os.path.exists(fpath):
            print(f"    {suffix}  ({os.path.getsize(fpath)/1e6:.1f} MB)")
        else:
            print(f"    {suffix}  [NOT FOUND]")

print(f"\n  Aggregate files:")
print(f"    {MODEL_PREFIX}_position_confound_by_layer.csv")
if N_SHUFFLES > 0:
    print(f"    {MODEL_PREFIX}_shuffle_decomposition_by_layer.csv")
for pm in POOLING_METHODS:
    for mn in ["no_social", "with_social"]:
        print(f"    {MODEL_PREFIX}_layer_margins_{pm}_{mn}.csv")

print("\n" + "=" * 60)
print("LOADING PATTERN FOR DOWNSTREAM NOTEBOOKS")
print("=" * 60)
print(f'''
# --- Stream A: Metrics (all boards, all permutations, no vectors) ---
metrics = pd.read_parquet(".../{MODEL_PREFIX}_metrics_no_social.parquet")
metrics_canonical = metrics[metrics["permutation_id"] == 0]

# --- Stream B: Vector subsample (canonical only) ---
index  = pd.read_csv(".../{MODEL_PREFIX}_vectors_subsample_index_no_social.csv")
data   = np.load(".../{MODEL_PREFIX}_vectors_subsample_no_social_f16.npz")
matrix = data["vectors"]   # shape [N, {{HIDDEN_DIM}}], dtype float16
vec    = matrix[i].astype(np.float32)

# Filter by pooling method:
idx_mean = index[index["pooling_method"] == "mean"]
idx_maxn = index[index["pooling_method"] == "max_norm"]

# --- Generation results (causal-only) ---
gen = pd.read_csv(".../{MODEL_PREFIX}_generation_no_social.csv")

# --- Shuffle analysis ---
shuffles = pd.read_csv(".../{MODEL_PREFIX}_shuffle_decomposition_by_layer.csv")
''')



OUTPUT SUMMARY
Directory: /content/drive/MyDrive/TCC/qwen_outputs

  [no_social]
    qwen_general_no_social.csv  (1.8 MB)
    qwen_metrics_no_social.parquet  (48.5 MB)
    qwen_vectors_subsample_index_no_social.csv  (4.6 MB)
    qwen_vectors_subsample_no_social_f16.npz  (367.6 MB)
    qwen_generation_no_social.csv  (0.1 MB)
  [with_social]
    qwen_general_with_social.csv  (3.1 MB)
    qwen_metrics_with_social.parquet  (57.6 MB)
    qwen_vectors_subsample_index_with_social.csv  (7.2 MB)
    qwen_vectors_subsample_with_social_f16.npz  (655.6 MB)
    qwen_generation_with_social.csv  (0.1 MB)

  Aggregate files:
    qwen_position_confound_by_layer.csv
    qwen_shuffle_decomposition_by_layer.csv
    qwen_layer_margins_mean_no_social.csv
    qwen_layer_margins_mean_with_social.csv
    qwen_layer_margins_max_norm_no_social.csv
    qwen_layer_margins_max_norm_with_social.csv

LOADING PATTERN FOR DOWNSTREAM NOTEBOOKS

# --- Stream A: Metrics (all boards, all permutations, no vectors) ---
metr